## dataloader

In [6]:
import sys

sys.path.insert(0, "../src")

import logging

from dataloader import SubjectWiseDataLoader, load_dataset

logging.basicConfig(level=logging.INFO)

# ── Method 1: Using the SubjectWiseDataLoader class ──────────────────────────
loader = SubjectWiseDataLoader()
splits = loader.prepare()

# Get train/val/test splits as DataFrames
train_df = loader.get_split("train")
val_df = loader.get_split("val")
test_df = loader.get_split("test")

print("\n" + "=" * 60)
print("DATALOADER SUMMARY")
print("=" * 60)
print(f"Features used: {loader.feature_cols}")
print(f"Target column: {loader.target_col}")

# Get as numpy arrays for model training
train_X, train_y = loader.get_split("train", return_arrays=True)
val_X, val_y = loader.get_split("val", return_arrays=True)
test_X, test_y = loader.get_split("test", return_arrays=True)

print(f"\nTrain: X shape {train_X.shape}, y shape {train_y.shape}")
print(f"Val:   X shape {val_X.shape}, y shape {val_y.shape}")
print(f"Test:  X shape {test_X.shape}, y shape {test_y.shape}")

print("\n" + "=" * 60)
print("Label distribution in each split:")
print("=" * 60)
for split_name in ["train", "val", "test"]:
    df = splits[split_name]
    label_counts = df["pseudo_label_smoothed"].value_counts().sort_index()
    print(f"{split_name}: {dict(label_counts)} (total: {len(df)})")

INFO:dataloader:Loaded dataset: 2541 rows, 31 columns
INFO:dataloader:Found 24 unique subjects
INFO:dataloader:Splitting 24 subjects into train/val/test...
INFO:dataloader:train: 2015 rows (79.3%), 18 subjects (75.0%), labels: {0: 1283, 1: 732}
INFO:dataloader:val: 48 rows (1.9%), 3 subjects (12.5%), labels: {0: 29, 1: 19}
INFO:dataloader:test: 478 rows (18.8%), 3 subjects (12.5%), labels: {0: 304, 1: 174}



DATALOADER SUMMARY
Features used: ['sdnn_znorm', 'cv_rr_znorm', 'rmssd_znorm', 'pnn50_znorm']
Target column: pseudo_label_smoothed

Train: X shape (2015, 4), y shape (2015,)
Val:   X shape (48, 4), y shape (48,)
Test:  X shape (478, 4), y shape (478,)

Label distribution in each split:
train: {0: np.int64(1283), 1: np.int64(732)} (total: 2015)
val: {0: np.int64(29), 1: np.int64(19)} (total: 48)
test: {0: np.int64(304), 1: np.int64(174)} (total: 478)


In [8]:
# ── Method 2: Using the convenience function ────────────────────────────────
import numpy as np

splits_alt, features = load_dataset()

print("\n" + "=" * 60)
print("ALTERNATIVE: Using load_dataset() convenience function")
print("=" * 60)
print(f"Features: {features}")
print(f"Train rows: {len(splits_alt['train'])}")
print(f"Val rows: {len(splits_alt['val'])}")
print(f"Test rows: {len(splits_alt['test'])}")

# ── Method 3: Get all splits as numpy arrays at once ────────────────────────
all_splits_arrays = loader.get_all_splits(return_arrays=True)

print("\n" + "=" * 60)
print("NUMPY ARRAYS: Ready for model training")
print("=" * 60)
for split_name, (X, y) in all_splits_arrays.items():
    print(f"{split_name}: X {X.shape} (float32), y {y.shape} (int64)")
    print(f"  -> y classes: {np.unique(y)}, class ratio: {np.bincount(y)}")

INFO:dataloader:Loaded dataset: 2541 rows, 31 columns
INFO:dataloader:Found 24 unique subjects
INFO:dataloader:Splitting 24 subjects into train/val/test...
INFO:dataloader:train: 2015 rows (79.3%), 18 subjects (75.0%), labels: {0: 1283, 1: 732}
INFO:dataloader:val: 48 rows (1.9%), 3 subjects (12.5%), labels: {0: 29, 1: 19}
INFO:dataloader:test: 478 rows (18.8%), 3 subjects (12.5%), labels: {0: 304, 1: 174}



ALTERNATIVE: Using load_dataset() convenience function
Features: ['sdnn_znorm', 'cv_rr_znorm', 'rmssd_znorm', 'pnn50_znorm']
Train rows: 2015
Val rows: 48
Test rows: 478

NUMPY ARRAYS: Ready for model training
train: X (2015, 4) (float32), y (2015,) (int64)
  -> y classes: [0 1], class ratio: [1283  732]
val: X (48, 4) (float32), y (48,) (int64)
  -> y classes: [0 1], class ratio: [29 19]
test: X (478, 4) (float32), y (478,) (int64)
  -> y classes: [0 1], class ratio: [304 174]


In [11]:
# noqa: E402, W293, E501
# ── Leave-One-Subject-Out (LOSO) Cross-Validation ────────────────────────

from dataloader import load_loso_folds

print("\n" + "=" * 60)
print("LOSO CROSS-VALIDATION")
print("=" * 60)

# Get LOSO folds
folds, features = load_loso_folds()
print(f"\nTotal folds: {len(folds)} (one subject left out per fold)")
print(f"Features: {features}")

# Display fold statistics
print("\nFold breakdown (first 5 and last 2):")
for fold_idx in list(range(5)) + list(range(len(folds) - 2, len(folds))):
    if fold_idx < len(folds):
        fold = folds[fold_idx]
        test_subj = fold["test"]["subject_id"].iloc[0]
        train_n = len(fold["train"])
        test_n = len(fold["test"])
        train_subj = fold["train"]["subject_id"].nunique()

        y_train = fold["train"]["pseudo_label_smoothed"]
        y_test = fold["test"]["pseudo_label_smoothed"]

        print(
            f"  Fold {fold_idx:2d}: left_out={test_subj:12s} | "
            f"train: {train_n:4d} rows ({train_subj} subj) | "
            f"test: {test_n:3d} rows | "
            f"train_labels: {dict(y_train.value_counts().sort_index())}"
        )

# Example: train and evaluate on all folds
print("\n" + "=" * 60)
print("USAGE EXAMPLE: Model Training Loop")
print("=" * 60)

print("\n# Template for cross-validation training:")
template = """
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score
)

folds, features = load_loso_folds()
metrics_per_fold = []

for fold_idx, fold in enumerate(folds):
    # Extract train/test data
    X_train = fold['train'][features].values
    y_train = fold['train']['pseudo_label_smoothed'].values
    X_test = fold['test'][features].values
    y_test = fold['test']['pseudo_label_smoothed'].values

    # Preprocess
    scaler = StandardScaler()
    X_train = scaler.fit_transform(X_train)
    X_test = scaler.transform(X_test)

    # Train and evaluate
    model = LogisticRegression()
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)

    # Metrics
    metrics = {
        'fold': fold_idx,
        'accuracy': accuracy_score(y_test, y_pred),
        'precision': precision_score(y_test, y_pred),
        'recall': recall_score(y_test, y_pred),
        'f1': f1_score(y_test, y_pred),
    }
    metrics_per_fold.append(metrics)
    print(f"Fold {fold_idx}: acc={metrics['accuracy']:.3f}, "
          f"f1={metrics['f1']:.3f}")

# Average across folds
import pandas as pd
results_df = pd.DataFrame(metrics_per_fold)
avg_acc = results_df['accuracy'].mean()
std_acc = results_df['accuracy'].std()
avg_f1 = results_df['f1'].mean()
std_f1 = results_df['f1'].std()
print(f"Average Accuracy: {avg_acc:.3f} ± {std_acc:.3f}")
print(f"Average F1-Score: {avg_f1:.3f} ± {std_f1:.3f}")
"""
print(template)


LOSO CROSS-VALIDATION


INFO:dataloader:Loaded dataset: 2541 rows, 31 columns
INFO:dataloader:Found 24 unique subjects
INFO:dataloader:Creating LOSO folds with 24 subjects...
INFO:dataloader:Fold  1/24: test_subject=ddd_01M, train_samples=2307, test_samples=234
INFO:dataloader:Fold  2/24: test_subject=ddd_02F, train_samples=2305, test_samples=236
INFO:dataloader:Fold  3/24: test_subject=ddd_03F, train_samples=2306, test_samples=235
INFO:dataloader:Fold  4/24: test_subject=ddd_04M, train_samples=2314, test_samples=227
INFO:dataloader:Fold  5/24: test_subject=ddd_05M, train_samples=2309, test_samples=232
INFO:dataloader:Fold  6/24: test_subject=ddd_06M, train_samples=2307, test_samples=234
INFO:dataloader:Fold  7/24: test_subject=ddd_07F, train_samples=2307, test_samples=234
INFO:dataloader:Fold  8/24: test_subject=ddd_08M, train_samples=2309, test_samples=232
INFO:dataloader:Fold  9/24: test_subject=ddd_09M, train_samples=2309, test_samples=232
INFO:dataloader:Fold 10/24: test_subject=ddd_10M, train_samples=23


Total folds: 24 (one subject left out per fold)
Features: ['sdnn_znorm', 'cv_rr_znorm', 'rmssd_znorm', 'pnn50_znorm']

Fold breakdown (first 5 and last 2):
  Fold  0: left_out=ddd_01M      | train: 2307 rows (23 subj) | test: 234 rows | train_labels: {0: np.int64(1480), 1: np.int64(827)}
  Fold  1: left_out=ddd_02F      | train: 2305 rows (23 subj) | test: 236 rows | train_labels: {0: np.int64(1489), 1: np.int64(816)}
  Fold  2: left_out=ddd_03F      | train: 2306 rows (23 subj) | test: 235 rows | train_labels: {0: np.int64(1467), 1: np.int64(839)}
  Fold  3: left_out=ddd_04M      | train: 2314 rows (23 subj) | test: 227 rows | train_labels: {0: np.int64(1480), 1: np.int64(834)}
  Fold  4: left_out=ddd_05M      | train: 2309 rows (23 subj) | test: 232 rows | train_labels: {0: np.int64(1453), 1: np.int64(856)}
  Fold 22: left_out=drozy_s13    | train: 2529 rows (23 subj) | test:  12 rows | train_labels: {0: np.int64(1608), 1: np.int64(921)}
  Fold 23: left_out=drozy_s14    | train: 252

## Classifier Training (XGBoost & CatBoost)

In [ ]:
%matplotlib inline
import matplotlib.pyplot as plt  # noqa: F401  # prime inline backend before train_classifiers
import pandas as pd
from IPython.display import Image, display
from sklearn.metrics import precision_recall_curve, roc_curve

from train_classifiers import (
    MODEL_NAMES,
    PLOTS_DIR,
    _format_comparison_fixed,
    _format_comparison_loso,
    _run_loso,
    _train_and_eval_fixed,
    plot_feature_importance,
    plot_pr_curves,
    plot_roc_curves,
)

### Fixed Split Training

Train XGBoost and CatBoost on the 80/10/10 subject-level split prepared above, then evaluate on val and test sets.

In [ ]:
# Reuse the `loader` already prepared in the dataloader cells above
fixed_results = _train_and_eval_fixed(loader)
print(_format_comparison_fixed(fixed_results))

In [ ]:
# Build curve data from test-set predictions and display ROC + PR curves
roc_data_fixed = {}
pr_data_fixed = {}
for name in MODEL_NAMES:
    y_true = fixed_results[name]["test"]["y_true"]
    y_proba = fixed_results[name]["test"]["y_proba"]
    fpr, tpr, _ = roc_curve(y_true, y_proba)
    prec, rec, _ = precision_recall_curve(y_true, y_proba)
    roc_data_fixed[name] = (fpr, tpr, fixed_results[name]["test"]["roc_auc"])
    pr_data_fixed[name] = (prec, rec, fixed_results[name]["test"]["pr_auc"])

plot_roc_curves(roc_data_fixed, suffix="_fixed")
plot_pr_curves(pr_data_fixed, suffix="_fixed")
display(Image(str(PLOTS_DIR / "fig_roc_curves_fixed.png")))
display(Image(str(PLOTS_DIR / "fig_pr_curves_fixed.png")))

In [ ]:
# Feature importance for both fitted models
fitted_models_fixed = {name: fixed_results[name]["model"] for name in MODEL_NAMES}
plot_feature_importance(fitted_models_fixed, loader.feature_cols)
display(Image(str(PLOTS_DIR / "fig_feature_importance.png")))

### Leave-One-Subject-Out Cross-Validation

Each of the 24 subjects is held out once. Predictions are pooled across all folds for aggregate evaluation. Folds where the test set contains only one class are skipped automatically.

In [ ]:
# `loader.df` is available after prepare() — no extra load_data() call needed
loso_results = _run_loso(loader)

In [ ]:
# Per-fold precision/recall/F1 for each model
for name in MODEL_NAMES:
    fold_df = pd.DataFrame(loso_results[name]["fold_records"])
    print(f"\n{name} — per-fold metrics:")
    cols = ["fold", "held_out_subject", "n_test", "precision", "recall", "f1", "roc_auc"]
    print(fold_df[cols].to_string(index=False))

# Aggregate metrics (pooled predictions at optimal threshold)
print("\n")
print(_format_comparison_loso(loso_results))

In [ ]:
# LOSO pooled ROC + PR curves
roc_data_loso = {}
pr_data_loso = {}
for name in MODEL_NAMES:
    r = loso_results[name]
    fpr, tpr, _ = roc_curve(r["y_true"], r["y_proba"])
    prec, rec, _ = precision_recall_curve(r["y_true"], r["y_proba"])
    roc_data_loso[name] = (fpr, tpr, r["overall"]["roc_auc"])
    pr_data_loso[name] = (prec, rec, r["overall"]["pr_auc"])

plot_roc_curves(roc_data_loso, suffix="_loso")
plot_pr_curves(pr_data_loso, suffix="_loso")
display(Image(str(PLOTS_DIR / "fig_roc_curves_loso.png")))
display(Image(str(PLOTS_DIR / "fig_pr_curves_loso.png")))

### Results Summary

Results are compared across two evaluation protocols:

- **Fixed Split**: 18 train / 3 val / 3 test subjects. Metrics at threshold = 0.5.
- **LOSO-CV**: 24 folds, pooled predictions. Threshold optimised for max precision subject to recall ≥ 0.30.

The code cell below identifies the better-performing model on each protocol.

In [ ]:
# Identify the better-performing model on each evaluation protocol
print("=== Fixed Split — Test Set ===")
best_fixed = max(MODEL_NAMES, key=lambda n: fixed_results[n]["test"]["f1"])
for name in MODEL_NAMES:
    m = fixed_results[name]["test"]
    print(
        f"  {name:10s}: Prec={m['precision']:.4f}  Rec={m['recall']:.4f}  "
        f"F1={m['f1']:.4f}  ROC-AUC={m['roc_auc']:.4f}  PR-AUC={m['pr_auc']:.4f}"
    )
print(f"\n  Best (F1): {best_fixed}")

print("\n=== LOSO-CV — Pooled (optimal threshold) ===")
best_loso = max(MODEL_NAMES, key=lambda n: loso_results[n]["overall"]["f1"])
for name in MODEL_NAMES:
    m = loso_results[name]["overall"]
    thr = loso_results[name]["opt_threshold"]
    print(
        f"  {name:10s}: Prec={m['precision']:.4f}  Rec={m['recall']:.4f}  "
        f"F1={m['f1']:.4f}  ROC-AUC={m['roc_auc']:.4f}  PR-AUC={m['pr_auc']:.4f}  "
        f"threshold={thr:.4f}"
    )
print(f"\n  Best (F1): {best_loso}")